In [ ]:
#importing important libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE

In [ ]:
#loading daatset
df = pd.read_csv("Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
print(df)

         Destination Port   Flow Duration   Total Fwd Packets  \
0                   54865               3                   2   
1                   55054             109                   1   
2                   55055              52                   1   
3                   46236              34                   1   
4                   54863               3                   2   
...                   ...             ...                 ...   
225740              61374              61                   1   
225741              61378              72                   1   
225742              61375              75                   1   
225743              61323              48                   2   
225744              61326              68                   1   

         Total Backward Packets  Total Length of Fwd Packets  \
0                             0                           12   
1                             1                            6   
2                          

In [ ]:
#striping column names
df.columns = df.columns.str.strip()

In [ ]:
#Deleting constant columns
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
print("\nDeleting constant columns:", constant_cols)

df.drop(columns=constant_cols, inplace=True)



Deleting constant columns: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


In [ ]:
#replacing infinite value with NaN
df = df.replace([np.inf, -np.inf], np.nan)

In [ ]:
#seprating categoricak and numerical value
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

In [ ]:
#filling missing values for numerical column
skew_values = df[num_cols].skew()

for col in num_cols:
    if abs(skew_values[col]) <= 0.5:
        # symmetrical → fill with mean
        df[col] = df[col].fillna(df[col].mean())
    else:
        # skewed → fill with median
        df[col] = df[col].fillna(df[col].median())

In [ ]:

#filling missing values for categorical column
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
#Encode Target Value

label_encoder = LabelEncoder()
df["Label_enc"] = label_encoder.fit_transform(df["Label"])

y = df["Label_enc"]
X = df.drop(columns=["Label", "Label_enc"])

In [ ]:
#Encoding categorical columns
for col in X.select_dtypes(include=["object"]).columns:
    X[col] = LabelEncoder().fit_transform(X[col])


In [ ]:
#Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
#Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)


In [ ]:
#SMOTE
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

In [ ]:
#Training and Evaluation of All Models
for name, model in models.items():
    print(f"\n================ {name} ================")

    # Train
    model.fit(X_train_res, y_train_res)

    # Predictions
    train_pred = model.predict(X_train_res)
    test_pred = model.predict(X_test)

    # Metrics
    train_acc = accuracy_score(y_train_res, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    f1 = f1_score(y_test, test_pred)

    print(f"Training Accuracy: {train_acc:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, test_pred))



================ Logistic Regression ================
Training Accuracy: 0.9986
Test Accuracy: 0.9987
F1 Score: 0.9988

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19544
           1       1.00      1.00      1.00     25605

    accuracy                           1.00     45149
   macro avg       1.00      1.00      1.00     45149
weighted avg       1.00      1.00      1.00     45149


================ Random Forest ================
Training Accuracy: 1.0000
Test Accuracy: 0.9999
F1 Score: 0.9999

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19544
           1       1.00      1.00      1.00     25605

    accuracy                           1.00     45149
   macro avg       1.00      1.00      1.00     45149
weighted avg       1.00      1.00      1.00     45149


================ Decision Tree ================
Training Accura